# Demo: Orquestador Multi-Agente (Supervisor + Investigador + Analista)

Este notebook demuestra el flujo completo de delegación:

1. El usuario hace una consulta que requiere **investigación** y **análisis**.
2. El **Supervisor** decide enviarla primero al **Agente de Investigación**.
3. El Investigador busca en la base simulada de pre-entregas (`data/pre_entregas_kb.json`) y devuelve hallazgos.
4. El Supervisor recibe el resultado, decide que falta análisis y delega al **Agente de Análisis**.
5. El Analista calcula el promedio de calificaciones y el sentimiento general de los comentarios.
6. El Supervisor evalúa el resultado contra su rúbrica; si es insuficiente, pide **un** refinamiento; si ya está OK, **finaliza**.

> Requiere una `GOOGLE_API_KEY` (gratuita, https://aistudio.google.com/app/apikey) en un archivo `.env` en esta carpeta (ver `.env.example`).

In [ ]:
from graph import app, route_from_supervisor
from langchain_core.messages import HumanMessage

## 1. Estructura del grafo (sin necesitar API key)

El grafo se puede inspeccionar y dibujar sin invocar ningún LLM, gracias a que los agentes se construyen de forma perezosa.

In [ ]:
print(app.get_graph().draw_mermaid())

## 2. Lógica anti-bucle-infinito (sin API key)

`decide_next_agent` es una función pura que garantiza que el grafo termina en como mucho `MAX_STEPS` pasos, sin importar lo que "opine" el LLM del Supervisor.

In [ ]:
from agents.supervisor import decide_next_agent

# Simulación: el LLM siempre "quiere refinar", pero el tope de refinamientos manda.
has_research = has_analysis = False
sufficient = False
refinements_used = 0
steps = 0
trace = []

for _ in range(20):
    next_agent = decide_next_agent(has_research, has_analysis, sufficient, refinements_used, steps, llm_wants_refine=True)
    trace.append((steps, next_agent))
    if next_agent == "FINISH":
        break
    if next_agent == "researcher":
        has_research = True
    elif next_agent == "analyst":
        if has_analysis:
            refinements_used += 1
        has_analysis = True
    steps += 1

trace

## 3. Ejecución real del flujo de delegación (requiere `GOOGLE_API_KEY` en `.env`)

In [ ]:
query = (
    "Investigá qué feedback recibieron las pre-entregas del curso relacionadas "
    "con RAG y extracción de entidades, y luego analizá el sentimiento general "
    "de los comentarios y el promedio de las calificaciones."
)

initial_state = {
    "messages": [HumanMessage(content=query)],
    "user_query": query,
    "next_agent": "researcher",
    "task_completed": False,
    "research_data": "",
    "analysis_data": "",
    "contributions": [],
    "steps": 0,
    "refinements_used": 0,
}

final_state = app.invoke(initial_state)

In [ ]:
for c in final_state["contributions"]:
    print(f"[paso {c['step']}] {c['agent']}: {c['summary']}\n")

In [ ]:
print("research_data:\n", final_state["research_data"])
print("\nanalysis_data:\n", final_state["analysis_data"])
print("\ntask_completed:", final_state["task_completed"])
print("refinements_used:", final_state["refinements_used"])

## 4. También se puede ver paso a paso con `.stream()`

In [ ]:
for chunk in app.stream(initial_state):
    for node, update in chunk.items():
        print(f"--- nodo: {node} ---")
        print({k: v for k, v in update.items() if k != "messages"})
        print()